# XGBOOST

In [ ]:
# Importer les bibliothèques nécessaires
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import yfinance as yf

## Téléchargement et traitement des données pour les 7 derniers jours

In [ ]:
def data_court_terme(t):
  ticker = yf.Ticker(t)
  var = ticker.history(period="7d", interval="1m")
  chemin = r"C:\Users\paulp\OneDrive - Groupe INSEEC (POCE)\Bureau\Market Finance\Machine Learning\\" + t + "_ct.csv"
  var.to_csv(chemin)
  df = pd.read_csv(chemin)

  df["Datetime"] = pd.to_datetime(df["Datetime"])

  df["Hour"] = df["Datetime"].dt.hour
  df["Minute"] = df["Datetime"].dt.minute
  df["Day"] = df["Datetime"].dt.dayofweek


  df["Hour_sin"] = np.sin(2 * np.pi * df["Hour"] / 7)
  df["Hour_cos"] = np.cos(2 * np.pi * df["Hour"] / 7)

  df["Minute_sin"] = np.sin(2 * np.pi * df["Minute"] / 7)
  df["Minute_cos"] = np.cos(2 * np.pi * df["Minute"] / 7)

  df["Day_sin"] = np.sin(2 * np.pi * df["Day"] / 5)
  df["Day_cos"] = np.cos(2 * np.pi * df["Day"] / 5)

  df = df.drop(columns = ["Hour", "Minute", "Day", "Dividends", "Stock Splits"])


  def RSI(df, period):
      """
      Add a Relative Strength Index column to a DataFrame

      Parameters:
      df : pandas DataFrame

      Returns:
      df_rsi : pandas DataFrame with an additional "RSI" column
      """
      # Calculate the daily change in closing price
      df['dif'] = df['Close'].diff()

      # Separate gains and losses
      df['gain'] = df['dif'].where(df['dif'] > 0, 0)
      df['loss'] = -df['dif'].where(df['dif'] < 0, 0)

      # Calculate the average gains and losses
      df['avg_gain'] = df['gain'].rolling(window=period, min_periods=1).mean()
      df['avg_loss'] = df['loss'].rolling(window=period, min_periods=1).mean()

      # Calculate the relative strength
      df['RS'] = df['avg_gain'] / df['avg_loss']

      # Calculate the RSI
      df['RSI'] = 100 - (100 / (1 + df['RS']))

      # Drop intermediate columns to keep the DataFrame clean
      df_rsi = df.drop(columns=['dif', 'gain', 'loss', 'avg_gain', 'avg_loss', 'RS'])

      return df_rsi


  df = RSI(df, 14)

  def PPO(df, short_period=12, long_period=26):
      """
      Add a PPO column to a DataFrame

      Parameters:
      df : pandas DataFrame
      short_period : The short period for the EMA calculation
      long_period : The long period for the EMA calculation

      Returns:
      df_ppo : pandas DataFrame with an additional "PPO" column
      """
      # Calculate the short-term EMA
      df['ema_short'] = df['Close'].ewm(span=short_period, adjust=False).mean()

      # Calculate the long-term EMA
      df['ema_long'] = df['Close'].ewm(span=long_period, adjust=False).mean()

      # Calculate the PPO
      df['PPO'] = ((df['ema_short'] - df['ema_long']) / df['ema_long']) * 100

      # Drop intermediate columns
      df_ppo = df.drop(columns=['ema_short', 'ema_long'])

      return df_ppo


  df = PPO(df)



  def MACD(df, short_period=12, long_period=26, signal_period=9):
      """
      Add a MACD column to a DataFrame

      Parameters:
      df : pandas DataFrame
      short_period : The short period for the MACD calculation
      long_period : The long period for the MACD calculation
      signal_period : The period for the signal line

      Returns:
      df_macd : pandas DataFrame with additional "MACD" and "Signal" columns
      """
      # Calculate the short-term exponential moving average (EMA)
      df['ema_short'] = df['Close'].ewm(span=short_period, adjust=False).mean()

      # Calculate the long-term exponential moving average (EMA)
      df['ema_long'] = df['Close'].ewm(span=long_period, adjust=False).mean()

      # Calculate the MACD line
      df['MACD'] = df['ema_short'] - df['ema_long']

      # Calculate the signal line
      df['Signal'] = df['MACD'].ewm(span=signal_period, adjust=False).mean()

      # Drop intermediate columns
      df_macd = df.drop(columns=['ema_short', 'ema_long'])

      return df_macd

  df = MACD(df)

  def OBV(df):
      """
      Add an On-Balance Volume column to a DataFrame

      Parameters:
      df : pandas DataFrame

      Returns:
      df_obv : pandas DataFrame with an additional "OBV" column
      """
      # Determine the price direction
      df['direction'] = (df['Close'] > df['Close'].shift(1)).astype(int) * 2 - 1
      df['direction'].iloc[0] = 0  # Set the direction of the first row to 0

      # Calculate the daily OBV change
      df['obv_change'] = df['direction'] * df['Volume']

      # Compute the cumulative OBV
      df['OBV'] = df['obv_change'].cumsum()

      # Drop intermediate columns
      df_obv = df.drop(columns=['direction', 'obv_change'])

      return df_obv


  df = OBV(df)


  def ROC(df, period):
      """
      Add a Rate of Change column to a DataFrame

      Parameters:
      df : pandas DataFrame
      period : The period over which the ROC is calculated

      Returns:
      df_roc : pandas DataFrame with an additional "ROC" column
      """
      # Calculate the price 'period' periods ago
      df['Close_period'] = df['Close'].shift(period)

      # Calculate the Rate of Change
      df['ROC'] = ((df['Close'] - df['Close_period']) / df['Close_period']) * 100

      # Drop intermediate columns
      df_roc = df.drop(columns=['Close_period'])

      return df_roc

  df = ROC(df, 14)

  df["RSI"] = df["RSI"].fillna(df["RSI"].mean())
  df["ROC"] = df["ROC"].fillna(df["ROC"].mean())
  df = df.rename(columns={"Datetime": "Date"})
  return df

## XGBoost (Court terme)

In [ ]:
df = data_court_terme("NVDA")

keep_ct = ['Open', 'High', 'Low', 'Volume', 'Hour_sin',
           'Hour_cos', 'Minute_sin', 'Minute_cos', 'Day_sin', 'Day_cos', 'RSI',
           'PPO', 'MACD', 'Signal', 'OBV', 'ROC']

# Normalisation des données
X = np.array(df.loc[:, keep_ct])
y = np.array(df["Close"]).reshape((-1, 1))
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X = scaler_X.fit_transform(X)
y = scaler_y.fit_transform(y)

# Séparation des données
test_size = int(len(X) * 0.1)
X_train, X_test = X[:-test_size], X[-test_size:]
y_train, y_test = y[:-test_size], y[-test_size:]

# Création du modèle XGBoost
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, max_depth=3)
xgb_model.fit(X_train, y_train)

#Enregistrement du model en binaire pour Quantconnect
xgb_model.save_model(r"C:\Users\paulp\OneDrive - Groupe INSEEC (POCE)\Bureau\Market Finance\Machine Learning\xgboost.bst")

# Prédictions
y_pred_xgb = xgb_model.predict(X_test)

# Calcul du RMSE pour XGBoost
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
print("XGBoost RMSE:", rmse_xgb)

# Inverse transformation pour ramener les valeurs dans l'échelle originale
y_test_inverse = scaler_y.inverse_transform(y_test)
y_pred_inverse = scaler_y.inverse_transform(y_pred_xgb.reshape(-1, 1))

# Affichage des courbes
plt.figure(figsize=(10, 4))
plt.plot(y_test_inverse, label="Valeurs Réelles", color='blue')
plt.plot(y_pred_inverse, label="Prédictions XGBoost", color='orange')
plt.title("Comparaison des prédictions XGBoost avec les vraies valeurs")
plt.xlabel("Échantillons")
plt.ylabel("Prix de clôture")
plt.legend()
plt.grid(True)
plt.show()